# Toy problem: $\delta$-function spectrum with self-similar peak

**Setup.** All the fluid kinetic energy is concentrated at a single (moving) wavenumber:

$$\boxed{\;E(k,t) \;=\; A(t)\,\delta\!\bigl(k - k_0(t)\bigr),\;}$$

with amplitude $A(t)$ and peak wavenumber $k_0(t)=1/L(t)$ following the self-similar laws
$$A(t) \;=\; u_0^2\,(1+t/\tau_\ast)^{-p},\qquad k_0(t) \;=\; k_{p,0}\,(1+t/\tau_\ast)^{-q},$$
with $q+p/2=1$ (the V.a closure: $L/u\sim t$). Loitsiansky: $(p,q)=(10/7,2/7)$. Saffman: $(p,q)=(6/5,2/5)$.

**Goal.** Get a fully analytical answer for

1. the frequency at which the GW source radiates at slow time $T$, $\omega_{\rm GW}(T)$,
2. how that peak frequency drifts as the source decays,
3. the GW spectrum $H_{ijij}(0,\omega)$ integrated over the emission window,

for the toy where the spectrum is a single $\delta$-shell.

Reuses the §5 K2008 aeroacoustic formula and the BK2016 temporal kernel from the parent notebook `decaying_selfsimilar_derivation.ipynb`. Same plot style hub for visual consistency.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import special, integrate

# numpy<2 compatibility
if not hasattr(np, 'trapezoid'):
    np.trapezoid = np.trapz

In [2]:
# =====================================================================
# Notebook-wide plotting style.  Edit THIS cell to restyle every figure.
# (Copied from decaying_selfsimilar_derivation.ipynb for self-containment.)
# =====================================================================

PALETTE = [
    '#0072B2', '#D55E00', '#009E73', '#CC79A7',
    '#E69F00', '#56B4E9', '#F0E442', '#000000',
]

plt.rcParams.update({
    'figure.figsize':      (8.0, 5.0),
    'figure.dpi':          110,
    'savefig.dpi':         150,
    'font.size':           15,
    'axes.titlesize':      17,
    'axes.labelsize':      16,
    'xtick.labelsize':     13,
    'ytick.labelsize':     13,
    'legend.fontsize':     13,
    'lines.linewidth':     2.6,
    'axes.linewidth':      1.4,
    'xtick.major.width':   1.4,
    'ytick.major.width':   1.4,
    'xtick.major.size':    6,
    'ytick.major.size':    6,
    'xtick.minor.visible': False,
    'ytick.minor.visible': False,
    'axes.grid':           False,
    'axes.prop_cycle':     plt.cycler(color=PALETTE),
    'legend.frameon':      False,
    'axes.spines.top':     False,
    'axes.spines.right':   False,
})

def tidy(ax, nx=5, ny=5):
    from matplotlib.ticker import MaxNLocator
    if ax.get_xscale() == 'linear':
        ax.xaxis.set_major_locator(MaxNLocator(nbins=nx))
    if ax.get_yscale() == 'linear':
        ax.yaxis.set_major_locator(MaxNLocator(nbins=ny))
    return ax

## 1. Setup and dimensional bookkeeping

### 1.1 Spectrum and scalings

Total kinetic energy: $\int_0^\infty E(k,t)\,dk = A(t) = u^2(t)$ — so $A$ literally *is* the (twice-) mean kinetic energy. Integral scale: $L(t)=1/k_0(t)$.

Self-similar laws (V.a closure $q+p/2=1$):

$$u(t)=u_0\,f^{-p/2},\qquad L(t)=L_0\,f^{q},\qquad k_0(t)=k_{p,0}\,f^{-q},\qquad f(t)\equiv 1+t/\tau_\ast.$$

Dissipation rate from $du^2/dt=-2\varepsilon$:
$$\varepsilon(t) = \frac{p\,u_0^2}{2\tau_\ast}\,f^{-(p+1)}.$$

### 1.2 The natural frequency: eddy turnover

A single-mode incompressible source at wavenumber $k_0$ with rms velocity $u$ has exactly **one** intrinsic timescale — the eddy turnover time

$$\tau_e(t) \;=\; \frac{L(t)}{u(t)} \;=\; \frac{1}{u(t)\,k_0(t)}.$$

The corresponding frequency $\omega_e(t) = u(t)\,k_0(t)$ is the rate at which a fluid element rotates around a vortex of scale $L$, and the rate at which the velocity field at $k_0$ rearranges itself.

By the V.a closure,

$$\omega_e(t) \;=\; u_0 k_{p,0}\,f^{-(p/2+q)} \;=\; \omega_{e,0}\,f^{-1},\qquad \omega_{e,0}\equiv u_0 k_{p,0}.$$

**The eddy frequency falls as $1/(1+t/\tau_\ast)$ — independent of which invariant class (Loitsiansky or Saffman) we picked.** The exponent $-1$ is the content of the self-similar closure, nothing else.

### 1.3 GW emission frequency from a single eddy

The Reynolds stress $T_{ij}\sim\rho\,u_i u_j$ is quadratic in the velocity, so a sinusoidal velocity at frequency $\omega_e$ produces a stress at frequencies $0$ and $2\omega_e$. The TT part of $T_{ij}$ that radiates GWs picks up the $2\omega_e$ component. So the GW source from a single eddy at slow time $T$ is centred on

$$\boxed{\;\omega_{\rm GW}(T) \;=\; 2\,\omega_e(T) \;=\; \frac{2\,\omega_{e,0}}{1+T/\tau_\ast}.\;}$$

**This is the GW peak frequency, and it sweeps from $2\omega_{e,0}$ at the start to $2\omega_{e,0}/(1+T_{\rm em}/\tau_\ast)$ at the end of the emission window.** Both extremes and the trajectory are class-independent.

## 2. From $E(k,t)$ to $H(0,\omega;T)$ — step by step, no black-boxes

The GW spectrum at long wavelengths ($k_{\rm GW}\!\to\!0$) is

$$H_{ijij}(0,\omega;T) \;=\; \int_{-\infty}^{\infty}\!d\tau\;e^{i\omega\tau}\;
\bigl\langle \Pi^{\rm TT}_{ij}(0,T+\tau/2)\,\Pi^{\rm TT}_{ij}{}^{\!*}(0,T-\tau/2)\bigr\rangle_{\!c}$$

where $\Pi^{\rm TT}_{ij}(\mathbf{k}_{\rm GW},t) = \int d^3x\,e^{-i\mathbf{k}_{\rm GW}\cdot\mathbf{x}}\,T_{ij}^{\rm TT}(\mathbf{x},t)$ and the cumulant subtracts the disconnected $\langle\Pi\rangle\langle\Pi^*\rangle$ (which is constant in $\tau$ and doesn't radiate). I will compute this from $E(k,t)$ alone, making every assumption explicit.

### 2.1 Velocity correlator from the spectrum

For homogeneous incompressible isotropic flow, the only piece of the Fourier two-point function consistent with these symmetries is

$$\bigl\langle\tilde u_i(\mathbf{k},t_1)\,\tilde u_j^{\,*}(\mathbf{k}',t_2)\bigr\rangle \;=\; (2\pi)^3\,\delta^3(\mathbf{k}-\mathbf{k}')\,P_{ij}(\hat{\mathbf{k}})\,\Phi(k;t_1,t_2),\qquad P_{ij}(\hat k)=\delta_{ij}-\hat k_i\hat k_j.$$

At coincident times, $\Phi(k;t,t) = E(k,t)/(4\pi k^2)$ — this is just the definition of $E$ as the shell-integrated kinetic energy density.

### 2.2 The piece we need to specify (this is *the* assumption)

For $t_1\ne t_2$, $E(k,t)$ alone does **not** determine $\Phi(k;t_1,t_2)$. We have to model how the velocity phase at wavenumber $k$ evolves. Parameterize as

$$\Phi(k;t_1,t_2) \;=\; \sqrt{\Phi(k,t_1)\,\Phi(k,t_2)}\;R(\tau;\,k,T),\qquad \tau=t_1-t_2,\;\;T=(t_1+t_2)/2,$$

with $R(0;k,T)=1$ and $|R|\le 1$ (Cauchy–Schwarz). The function $R$ encodes the temporal model — and it is what distinguishes the *broadband random-phase* picture from the *coherent eddy* picture.

For a $\delta$-spectrum, all the dynamics happens at one scale $k=k_0(T)$, so $R$ depends only on $\tau$ and $T$. Two natural choices:

| Model | $R(\tau;T)$ for $\sigma\equiv\tau/\tau_e(T)$ | Physical interpretation |
|---|---|---|
| (A) Random-phase BK2016 | $(1+|\sigma|)^{-2/3}$ | many modes on the shell $|\mathbf{k}|=k_0$, random phases, no oscillation — what we used before |
| (B) Coherent eddy | $\cos\sigma\,\,(\equiv\cos(\omega_e\tau))$ | one phase-coherent mode oscillating at $\omega_e = u\cdot k_0 = 1/\tau_e$ |

For a literal $\delta$-spectrum where *all* the energy sits in one mode, (B) is the more natural minimal model. (A) is the standard turbulence model; it averages over many modes assumed to be uncorrelated, which strictly contradicts the $\delta$-spectrum assumption. **Below we carry both and compare.**

### 2.3 Stress correlator via Wick's theorem

Under the Gaussian assumption (the *only* place Gaussianity enters), the connected stress correlator factorises:

$$\bigl\langle u_iu_j(\mathbf{x},t_1)\,u_ku_l(\mathbf{x}',t_2)\bigr\rangle_{\!c} \;=\; \bigl\langle u_iu_k\bigr\rangle\bigl\langle u_ju_l\bigr\rangle + \bigl\langle u_iu_l\bigr\rangle\bigl\langle u_ju_k\bigr\rangle.$$

Fourier-transform in $\mathbf{x},\mathbf{x}'$ and set the GW wavenumber to zero. The Gaussian product of two velocity correlators yields, after the standard TT contraction (the $\tfrac{14}{3}$ from spherical integration of two transverse projectors):

$$\bigl\langle\Pi^{\rm TT}\Pi^{\rm TT}{}^{\!*}\bigr\rangle_{\!c}(\mathbf{k}=0;\,\tau,T) \;=\; \frac{14}{3}\,\rho^2\,\mathcal{V}\,\int\!\frac{d^3q}{(2\pi)^3}\;\Phi^{\,2}(q;\,\tau,T),$$

where $\mathcal{V}$ is the (regulating) volume. So far no shape of $E$ has been assumed — only Gaussian closure.

### 2.4 Insert the delta-shell

Regularise the delta as a thin shell of width $\Delta\ll k_0$:

$$\Phi(q;t) \;=\; \frac{A(t)}{4\pi q^2}\,\frac{1}{\Delta}\,\mathbf{1}_{|q-k_0(t)|<\Delta/2}.$$

Squaring and integrating $\int d^3q/(2\pi)^3 = (4\pi q^2/(2\pi)^3)\int dq$ over the shell:

$$\int\!\frac{d^3q}{(2\pi)^3}\,\Phi^{\,2}(q;\tau,T) \;=\; \frac{A^2(T)}{32\pi^4\,k_0^2(T)\,\Delta}\;R^2(\tau;T) \;+\; O(\Delta).$$

(The two shells at $k_0(t_1)$ and $k_0(t_2)$ overlap for $|t_1-t_2|<\Delta(\tau_\ast+T)/(q\,k_0)$, which is *longer* than $\tau_e$ for any $\Delta\gtrsim q\,\epsilon_0/k_0$ — i.e., the shell-motion decorrelation is negligible compared to $R$, so we keep the slow-time $A^2$ evaluated at the midpoint $T$.)

### 2.5 Boxed result for the delta-spectrum

Putting it together,

$$\boxed{\;H_{ijij}(0,\omega;T) \;=\; \mathcal{C}\;\frac{A^2(T)}{k_0^2(T)}\;\tau_e(T)\;\mathcal{G}\!\bigl(\omega\,\tau_e(T)\bigr),
\quad\mathcal{C}=\frac{7\,\rho^2\mathcal{V}}{48\,\pi^5\,\Delta},\;}$$

where the **dimensionless temporal kernel**

$$\mathcal{G}(q) \;\equiv\; \frac{1}{\tau_e}\int_{-\infty}^{\infty}\!d\tau\;e^{i\omega\tau}\,R^2(\tau;T) \;=\; \int_{-\infty}^{\infty}\!d\sigma\;e^{iq\sigma}\,R^2(\sigma)$$

is **the FT of $R^2$**, evaluated at the dimensionless frequency $q=\omega\tau_e(T)$. *Everything* about the GW frequency content lives in $\mathcal{G}(q)$, and $\mathcal{G}$ is determined entirely by the choice of $R$ in §2.2.

## 3. The two kernels, analytically

### 3.1 Model A — random-phase BK2016

$R_A(\sigma) = (1+|\sigma|)^{-2/3}$, so $R_A^2(\sigma)=(1+|\sigma|)^{-4/3}$.

$$\mathcal{G}_A(q) \;=\; 2\,{\rm Re}\!\int_0^\infty\!d\sigma\,e^{iq\sigma}(1+\sigma)^{-4/3}.$$

This is monotonically decreasing in $|q|$:
- $\mathcal{G}_A(0) = 2\int_0^\infty(1+\sigma)^{-4/3}d\sigma = 2\cdot 3 = 6$ (finite).
- $\mathcal{G}_A(q\to\infty) \sim |q|^{-2/3}$ (from the corner at $\sigma=0$; weaker tail than the previous $G_{\rm BK}$ because $R^2$ has weaker corner).

**No peak at $\omega=2\omega_e$.** All the GW power sits at $\omega\lesssim 1/\tau_e=\omega_e$.

### 3.2 Model B — coherent eddy

$R_B(\sigma) = \cos\sigma$, so $R_B^2(\sigma) = \cos^2\sigma = \tfrac{1}{2} + \tfrac{1}{2}\cos(2\sigma)$.

$$\mathcal{G}_B(q) \;=\; \int_{-\infty}^{\infty}\!d\sigma\,e^{iq\sigma}\Bigl[\tfrac{1}{2} + \tfrac{1}{2}\cos(2\sigma)\Bigr]
\;=\; \pi\,\delta(q) \;+\; \tfrac{\pi}{2}\bigl[\delta(q-2)+\delta(q+2)\bigr].$$

**Three sharp peaks: a DC line at $\omega=0$ and a pair of side lines at $\omega=\pm 2\omega_e$.** The two side peaks together carry half the spectral weight, the DC carries the other half — the $\tfrac{1}{2}+\tfrac{1}{2}\cos(2\sigma)$ decomposition.

In the original frequency variable, the high-frequency peaks of $H(0,\omega;T)$ sit at

$$\boxed{\;\omega \;=\; \pm 2\,\omega_e(T) \;=\; \pm\frac{2\,\omega_{e,0}}{1+T/\tau_\ast}\;}$$

— exactly the eddy-rotation-doubled frequency, drifting with $T$ along the closure trajectory $1/(1+T/\tau_\ast)$ from §1.

### 3.3 Finite lifetime smooths the sharp peaks

In practice the coherent oscillation has a finite lifetime $\tau_L \sim$ a few $\tau_e$. Replace

$$R_B(\sigma) \;\to\; \cos\sigma\;e^{-\alpha|\sigma|},\qquad \alpha\equiv \tau_e/\tau_L.$$

Then $R_B^2 = e^{-2\alpha|\sigma|}\bigl[\tfrac{1}{2}+\tfrac{1}{2}\cos(2\sigma)\bigr]$, and the FT gives three Lorentzians of width $2\alpha$ centred at $q=0,\pm2$. For $\alpha\ll 1$ the peaks are well-separated; for $\alpha\to 0$ they recover the deltas.

This is what we'll plot below. $\alpha$ is the *one* remaining model parameter — the coherence time in units of the oscillation period.

In [ ]:
# Compute the two kernels G_A(q) (BK2016 random-phase) and G_B(q) (coherent eddy)
# directly from R^2, by FT. Both follow the boxed formula G = FT[R^2].

def G_random_phase(q, sigma_max=300.0, n=10001):
    """Model A: R = (1+|sigma|)^{-2/3}, so G_A(q) = FT of (1+|sigma|)^{-4/3}."""
    sig = np.linspace(0.0, sigma_max, n)
    return 2.0 * np.trapezoid(np.exp(1j*q*sig) * (1.0 + sig)**(-4/3), sig).real

def G_coherent(q, alpha=0.15, sigma_max=200.0, n=20001):
    """Model B: R = cos(sigma) * exp(-alpha |sigma|), so R^2 = e^(-2 alpha |sigma|) [1/2 + (1/2) cos(2 sigma)].
       Analytic form: three Lorentzians at q = 0, +/- 2, each of width 2 alpha."""
    a = 2.0 * alpha
    return (a / (a**2 + q**2)
            + 0.5 * a / (a**2 + (q-2)**2)
            + 0.5 * a / (a**2 + (q+2)**2)) * 2.0  # overall factor: int of e^{-a|s|} = 2/a, scaled

# Quick consistency check at q=0:
print(f"G_A(0)  (BK2016 random-phase)      = {G_random_phase(0.0):.3f}    (analytic 6)")
print(f"G_B(0)  (coherent, alpha=0.15)     = {G_coherent(0.0):.3f}")
print(f"G_B(2)  (coherent peak at 2 omega_e) = {G_coherent(2.0):.3f}")
print(f"G_B(5)  (coherent, far from peaks) = {G_coherent(5.0):.4f}")


In [ ]:
# Plot G_A and G_B side by side.

qs = np.linspace(-5, 5, 800)
GA = np.array([G_random_phase(q) for q in qs])
GB = np.array([G_coherent(q, alpha=0.15) for q in qs])

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(qs, GA, color=PALETTE[0], label=r"Model A  (random-phase BK2016)")
ax.plot(qs, GB, color=PALETTE[1], label=r"Model B  (coherent eddy, $\alpha=0.15$)")
ax.axvline( 2, color=PALETTE[3], ls=':', lw=1.3, alpha=0.7)
ax.axvline(-2, color=PALETTE[3], ls=':', lw=1.3, alpha=0.7)
ax.set_xlabel(r"$q = \omega\,\tau_e$")
ax.set_ylabel(r"$\mathcal{G}(q)$")
ax.set_title(r"Dimensionless GW source kernel  $\mathcal{G}(q) = \int d\sigma\,e^{iq\sigma}\,R^2(\sigma)$")
ax.legend()
tidy(ax)
plt.tight_layout(); plt.savefig('/tmp/kernels.png', bbox_inches='tight', dpi=110)
plt.show()
print("Model A: broad bump at q=0, no side peaks.")
print("Model B: DC peak at q=0 + side peaks at q=+/-2 (= +/- 2 omega_e).")


## 4. Where the GW power sits as the source decays

The closure $q+p/2=1$ forces (independently of class):

$$\omega_e(T) \;=\; \frac{\omega_{e,0}}{1+T/\tau_\ast},\qquad \tau_e(T) \;=\; \tau_{e,0}\,(1+T/\tau_\ast).$$

So in the original frequency $\omega$, the peak of $\mathcal{G}(\omega\tau_e(T))$ at $q=2$ sits at

$$\omega_{\rm peak}(T) \;=\; \frac{2}{\tau_e(T)} \;=\; \frac{2\,\omega_{e,0}}{1+T/\tau_\ast},$$

drifting from $2\omega_{e,0}$ at the start to $2\omega_{e,0}/(1+T_{\rm em}/\tau_\ast)$ at the end of emission.

The amplitude prefactor in §2.5 is $A^2(T)\,\tau_e(T)/k_0^2(T)$. Using $A=u^2$ and $k_0=1/L$:

$$\frac{A^2\,\tau_e}{k_0^2} \;=\; u^4\,L^2\cdot\frac{L}{u} \;=\; u^3\,L^3 \;\propto\; (1+T/\tau_\ast)^{-3p/2+3q}.$$

For Loitsiansky $(-3p/2+3q=-9/7)$ the prefactor decays as $f^{-9/7}$; for Saffman $f^{-3/5}$. The peak frequency drifts the *same way* in both classes; only the amplitude decays at a different rate.

In [ ]:
# Self-similar pieces (Loitsiansky default).
CLASSES = {'Loitsiansky': dict(p=10/7, q=2/7), 'Saffman': dict(p=6/5, q=2/5)}

def f_of_T(T, tau_st=1.0):    return 1.0 + T/tau_st
def omega_e(T, tau_st=1.0):   return 1.0 / f_of_T(T, tau_st)   # in units of omega_{e,0}
def tau_e(T, tau_st=1.0):     return f_of_T(T, tau_st)         # in units of tau_{e,0}

def H_inst(omega, T, model, p, q, alpha=0.15, tau_st=1.0):
    """Instantaneous H(0, omega; T) up to the overall constant C.
       'model' = 'A' (random-phase) or 'B' (coherent eddy)."""
    f  = f_of_T(T, tau_st)
    te = tau_e(T, tau_st)
    prefactor = (1.0)**3 * (1.0/1.0)**3 * f**(-3*p/2 + 3*q)   # u^3 L^3 (units u_0=1, L_0=1)
    qq = omega * te
    G = G_random_phase(qq) if model == 'A' else G_coherent(qq, alpha=alpha)
    return prefactor * te * G


In [ ]:
# Plot 1: instantaneous H(0, omega; T) for a few T, both models side by side.

p, q = CLASSES['Loitsiansky']['p'], CLASSES['Loitsiansky']['q']
omegas = np.linspace(0, 4.0, 800)
T_snap = [0.0, 0.5, 2.0, 5.0]

fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for T in T_snap:
    HA = np.array([H_inst(om, T, 'A', p, q) for om in omegas])
    HB = np.array([H_inst(om, T, 'B', p, q) for om in omegas])
    axA.plot(omegas, HA, label=fr"$T/\tau_\ast={T:.1f}$")
    axB.plot(omegas, HB, label=fr"$T/\tau_\ast={T:.1f}$")
    axA.axvline(omega_e(T),   color='gray', ls=':', lw=0.8, alpha=0.6)
    axB.axvline(2*omega_e(T), color='gray', ls=':', lw=0.8, alpha=0.6)
axA.set_title("Model A: random-phase BK2016")
axA.set_xlabel(r"$\omega/\omega_{e,0}$"); axA.set_ylabel(r"$H(0,\omega;T)$  (arb.)")
axB.set_title(r"Model B: coherent eddy ($\alpha=0.15$)")
axB.set_xlabel(r"$\omega/\omega_{e,0}$")
axA.legend(); tidy(axA); tidy(axB)
plt.tight_layout(); plt.savefig('/tmp/instant_spectra.png', bbox_inches='tight', dpi=110)
plt.show()


## 5. Animation — the delta peak and the GW source moving together

Three things move with $T$:

- The $E(k,T)$ peak at $k_0(T) = (1+T/\tau_\ast)^{-q}$  (slow drift, class-dependent).
- The Model-A knee at $\omega_e(T) = (1+T/\tau_\ast)^{-1}$  (faster drift, class-independent).
- The Model-B peak at $2\,\omega_e(T) = 2/(1+T/\tau_\ast)$ (twice the knee, same trajectory shape).

Both Models A and B are shown on the same axes so the difference is direct: Model B has a real sharp peak at $2\omega_e(T)$; Model A has only a broadband fall-off without any peak structure.

In [ ]:
# Animation: E(k,T) delta peak (top) and the two GW source models (bottom),
# all on the same figure. Saves /tmp/delta_anim_v2.gif.

import matplotlib.animation as animation
from IPython.display import HTML

p, q = CLASSES['Loitsiansky']['p'], CLASSES['Loitsiansky']['q']

T_frames = np.concatenate([np.linspace(0.0, 1.5, 25), np.linspace(1.5, 8.0, 35)])

ks_grid = np.geomspace(0.05, 3.0, 250)
om_grid = np.linspace(0.0, 3.5, 600)

SIGMA_LOG = 0.05
def E_show(k, T):
    k0 = (1.0 + T)**(-q)
    A  = (1.0 + T)**(-p)
    return A * np.exp(-(np.log(k/k0))**2/(2*SIGMA_LOG**2))

E_data = [E_show(ks_grid, T) for T in T_frames]
HA_data = [np.array([H_inst(om, T, 'A', p, q) for om in om_grid]) for T in T_frames]
HB_data = [np.array([H_inst(om, T, 'B', p, q) for om in om_grid]) for T in T_frames]

# Normalise each to its T=0 max so all three fit one axis
E_data  = [e / E_data[0].max()  for e in E_data]
HA_data = [h / HA_data[0].max() for h in HA_data]
HB_data = [h / HB_data[0].max() for h in HB_data]

fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(11, 8))

(line_E,)  = ax_top.semilogy(ks_grid, E_data[0],  color=PALETTE[0], lw=3.0,
                              label=r"$E(k,T)$  (delta peak at $k_0(T)$)")
peak_E_v   = ax_top.axvline(1.0, color=PALETTE[0], ls=':', lw=1.2)
ax_top.set_xlim(ks_grid[0], ks_grid[-1])
ax_top.set_ylim(1e-3, 2.0)
ax_top.set_xlabel(r"$k/k_{p,0}$")
ax_top.set_ylabel("energy spectrum (norm)")
ax_top.legend(loc='upper right')

(line_HA,) = ax_bot.plot(om_grid, HA_data[0], color=PALETTE[1], lw=2.8,
                          label=r"Model A: random-phase BK2016")
(line_HB,) = ax_bot.plot(om_grid, HB_data[0], color=PALETTE[2], lw=2.8,
                          label=r"Model B: coherent eddy")
knee_A = ax_bot.axvline(1.0, color=PALETTE[1], ls=':', lw=1.2)
peak_B = ax_bot.axvline(2.0, color=PALETTE[2], ls=':', lw=1.2)
ax_bot.set_xlim(0, 3.5)
ax_bot.set_ylim(0, 1.4)
ax_bot.set_xlabel(r"$\omega/\omega_{e,0}$")
ax_bot.set_ylabel("GW source $H(0,\omega;T)$ (norm)")
ax_bot.legend(loc='upper right')

title = fig.suptitle(r"$T/\tau_\ast = 0.00$    $k_0/k_{p,0}=1.000$    $\omega_e/\omega_{e,0}=1.000$",
                     fontsize=15)

def update(i):
    T = T_frames[i]
    line_E.set_ydata(E_data[i])
    line_HA.set_ydata(HA_data[i])
    line_HB.set_ydata(HB_data[i])
    k0 = (1.0 + T)**(-q)
    we = 1.0 / (1.0 + T)
    peak_E_v.set_xdata([k0, k0])
    knee_A.set_xdata([we, we])
    peak_B.set_xdata([2*we, 2*we])
    title.set_text(fr"$T/\tau_\ast={T:.2f}$    $k_0/k_{{p,0}}={k0:.3f}$    "
                   fr"$\omega_e/\omega_{{e,0}}={we:.3f}$    "
                   fr"(Model B peak at $\omega/\omega_{{e,0}}={2*we:.3f}$)")
    return line_E, line_HA, line_HB, peak_E_v, knee_A, peak_B, title

anim = animation.FuncAnimation(fig, update, frames=len(T_frames), interval=80, blit=False)
anim.save('/tmp/delta_anim_v2.gif', writer='pillow', fps=12)
print("saved /tmp/delta_anim_v2.gif")
plt.close(fig)

HTML(anim.to_jshtml())


## 6. Summary

**Spectrum alone doesn't fix the GW spectrum.** We had to pick how the phase at the single mode evolves — random-phase decorrelation (Model A) vs coherent oscillation (Model B).

**Boxed kernel formula** (valid for any $R$): $H(0,\omega;T) \;\propto\; \dfrac{A^2(T)\,\tau_e(T)}{k_0^2(T)}\,\mathcal{G}(\omega\tau_e(T))$ with $\mathcal{G}(q)=\text{FT}_\sigma[R^2(\sigma)]$.

**Model A (random-phase BK2016).** $R=(1+|\sigma|)^{-2/3}$. $\mathcal{G}_A$ is a broad bump peaked at $q=0$; the GW knee is at $\omega\sim\omega_e(T)$ and drifts as $1/(1+T/\tau_\ast)$. *No peak at $2\omega_e$.*

**Model B (coherent eddy).** $R=\cos\sigma\,e^{-\alpha|\sigma|}$. $\mathcal{G}_B$ has a DC line at $q=0$ and sharp side peaks at $q=\pm 2$. The high-frequency peaks of $H(0,\omega;T)$ sit at $\omega=\pm 2\omega_e(T)$ and drift as $\pm 2/(1+T/\tau_\ast)$ — exactly the eddy-frequency-doubled scaling.

**Which one is right for a $\delta$-spectrum?** Model B is the *minimal* assumption consistent with "all energy in one phase-coherent mode". Model A assumes random phases on the shell — physically inconsistent with $E=A\delta(k-k_0)$ (which is the limit of a *single* mode, not an ensemble), but standard in turbulence-GW literature.

Take-home: the previous broadband result was a consequence of importing the random-phase decorrelation. The literal coherent-mode reading of the $\delta$-spectrum gives a sharp moving peak at $2\omega_e(T)$, *exactly* as the dimensional eddy-frequency argument suggested.